# 05 · Record features (prefix `r_`)

> **Status (2026-11-25):** run on the **cleaned** pairs (`pairs_clean.parquet`), Vova's train + val only (253,955 pairs, 74.5% positive). All conclusions and findings use these numbers. The earlier run on the uncleaned pairs gave the same decisions.

Features from the record fields besides the name: birth date, country / nationality, ID number and address. Explored here; the chosen features are now in `src/screening/record_features.py` and are written by `scripts/03b_record_features.py` (same contract as the name features). The code there is the version that counts; this notebook keeps the evidence.

- **Input:** `config.PAIRS_INPUT` (the cleaned pairs), restricted to Vova's **train + val** pairs (`pairs_split.parquet`), so his test set stays untouched.
- **Rule for every feature:** compare the two sides (same / different / missing, or a similarity), never pass a raw value (a raw birth date or country would let the model recognise entities).
- **Leakage test for every feature:** could it describe the labelling process rather than the entities? Notebook 06 showed that the pattern of which fields are present already predicts the label (AUC 0.79), so the "missing" categories get their own check (section 5).
- **Split for the model check:** by person cluster (IDs linked by positive pairs), the split agreed with Vova (notebook 07, section 10).

Abbreviations: **FA rate** = false-alarm rate (share of negative pairs with an alarm); **pp** = percentage points; **AUC** = area under the ROC curve (0.5 = no signal); **leak strength** = max(AUC, 1 − AUC).

In [ ]:
import sys, re, time, hashlib
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from screening import config as C
from screening.lists import values
from screening.name_features import normalize

pd.set_option("display.width", 160)
RECALLS, TARGET = [0.95, 0.98, 0.99, 0.995], 0.99
props = ["birthDate", "country", "nationality", "idNumber", "address", "topics"]
cols = ["pair_id", "left_id", "right_id", "judgement", "left_schema"] + [f"{s}_{p}" for s in ("left", "right") for p in props]
pairs = pd.read_parquet(C.PAIRS_INPUT, columns=cols)
pairs = pairs[pairs["judgement"].isin(["positive", "negative"])].reset_index(drop=True)
pairs["y"] = (pairs["judgement"] == "positive").astype(int)
pairs["person"] = pairs["left_schema"] == "Person"
vsplit = pd.read_parquet(C.PAIRS_SPLIT, columns=["pair_id", "split"])
pairs = pairs.merge(vsplit, on="pair_id", how="inner", validate="one_to_one")
print("Vova's split:", pairs["split"].value_counts().to_dict(), "(test is dropped)")
pairs = pairs[pairs["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"input {C.PAIRS_INPUT.name}: {len(pairs):,} labelled pairs, {pairs['y'].mean():.1%} positive")

## 1 · Feature definitions

| Feature | Values | How |
| --- | --- | --- |
| `r_dob_status` | exact, partial_match, same_year, close, different, missing | best relation over all birth dates of both sides. **exact** = same full date; **partial_match** = equal as far as both are known, at least one side only year or year-month (1968 vs 1968-03-02); **same_year** = full dates, same year, other month/day; **close** = years differ by 1; **different** = otherwise; **missing** = no birth date on at least one side |
| `r_dob_year_gap` | number (years) | smallest absolute difference between the birth years; NaN if missing |
| `r_country_status` | same, different, missing | country and nationality together: same = at least one code on both sides |
| `r_id_status` | same, different, missing | ID numbers reduced to letters and digits (lower case, at least 4 characters); same = equal, or one contains the other (at least 6 characters, formatting differences); different = both sides have IDs, none match |
| `r_address_sim` | 0–1 | best token-set similarity over the addresses of both sides (at most 10 per side); NaN if missing |
| `x_topics_jaccard` | 0–1 | overlap of the topic tags (sanction, role.pep, …). **Prefix `x_`: candidate only, not in the feature set**, see the leakage check |

In [ ]:
DOB_RE = re.compile(r"^(\d{4})(?:-(\d{2}))?(?:-(\d{2}))?")
def parse_dob(s):
    m = DOB_RE.match(str(s).strip())
    return tuple(int(x) for x in m.groups() if x) if m else None

DOB_ORDER = ["exact", "partial_match", "same_year", "close", "different"]
def dob_relation(a, b):
    if len(a) == 3 and a == b:
        return "exact"
    k = min(len(a), len(b))
    if a[:k] == b[:k]:
        return "partial_match"
    if a[0] == b[0]:
        return "same_year"
    if abs(a[0] - b[0]) == 1:
        return "close"
    return "different"

def dob_features(r):
    L = [d for d in map(parse_dob, values(r, "left", ["birthDate"])) if d]
    R = [d for d in map(parse_dob, values(r, "right", ["birthDate"])) if d]
    if not L or not R:
        return "missing", np.nan
    rels = {dob_relation(a, b) for a in L for b in R}
    return next(x for x in DOB_ORDER if x in rels), float(min(abs(a[0] - b[0]) for a in L for b in R))

def set_status(r, props_):
    L = {str(v).strip().lower() for v in values(r, "left", props_)} - {""}
    R = {str(v).strip().lower() for v in values(r, "right", props_)} - {""}
    if not L or not R:
        return "missing"
    return "same" if L & R else "different"

_ID = re.compile(r"[^a-z0-9]")
def id_status(r):
    norm = lambda side: {x for x in (_ID.sub("", str(v).lower()) for v in values(r, side, ["idNumber"])) if len(x) >= 4}
    L, R = norm("left"), norm("right")
    if not L or not R:
        return "missing"
    if L & R or any((a in b or b in a) and min(len(a), len(b)) >= 6 for a in L for b in R):
        return "same"
    return "different"

def address_sim(r):
    L = [x for x in (normalize(v) for v in values(r, "left", ["address"])) if x][:10]
    R = [x for x in (normalize(v) for v in values(r, "right", ["address"])) if x][:10]
    if not L or not R:
        return np.nan
    return process.cdist(L, R, scorer=fuzz.token_set_ratio).max() / 100

def topics_jaccard(r):
    L, R = set(values(r, "left", ["topics"])), set(values(r, "right", ["topics"]))
    if not L or not R:
        return np.nan
    return len(L & R) / len(L | R)

def record_features(r):
    dob, gap = dob_features(r)
    return {"pair_id": r.pair_id, "r_dob_status": dob, "r_dob_year_gap": gap,
            "r_country_status": set_status(r, ["country", "nationality"]),
            "r_id_status": id_status(r), "r_address_sim": address_sim(r), "x_topics_jaccard": topics_jaccard(r)}

# quick checks on hand-made values
print(dob_relation((1968,), (1968, 3, 2)), dob_relation((1968, 3, 2), (1968, 2, 3)), dob_relation((1968,), (1969,)))

R_CATS = {"r_dob_status": DOB_ORDER + ["missing"], "r_country_status": ["same", "different", "missing"],
          "r_id_status": ["same", "different", "missing"]}
t0 = time.time()
rf = pd.DataFrame([record_features(r) for r in pairs.itertuples(index=False)])
for c, cats in R_CATS.items():
    rf[c] = pd.Categorical(rf[c], categories=cats)
print(f"{len(rf):,} pairs in {time.time() - t0:.0f}s")

## 2 · What the features look like

### Table: categories and share of positive pairs

**What it shows:** for each categorical record feature, how many pairs fall into each category (persons and companies together), the share of all pairs and the share of positive pairs in that category.

**How to read it:** a category far from the overall 74.5% positive carries signal. "exact" birth date should be almost always positive, "different" mostly negative. Watch the "missing" rows: if their share of positives differs a lot from the average, missingness itself carries signal (section 5).

In [ ]:
d = rf.merge(pairs[["pair_id", "y", "person"]], on="pair_id")
tabs = []
for c in R_CATS:
    t = d.groupby(c, observed=False).agg(pairs=("y", "size"), share_positive=("y", "mean"))
    t["share_of_pairs"] = t["pairs"] / len(d)
    t.index = pd.MultiIndex.from_product([[c], t.index.astype(str)], names=["feature", "value"])
    tabs.append(t)
pd.concat(tabs)[["pairs", "share_of_pairs", "share_positive"]].round(3)

**Conclusion.**
- **Strong in both directions:** an exact birth date (99.0% positive) or the same ID number (99.1%), and a different birth date (1.7%) or different ID numbers (1.3%), almost decide the label.
- **Country helps mainly through "different"** (4.0% positive); "same country" (68.2%) is below the average of 74.5%, because many confusions are within one country.
- **"missing" is not neutral:** pairs without a country are 94.3% positive, pairs without a birth date 64.0%. Missingness carries signal, mainly for the country.

**Next step:** check how much of the country signal is presence only (section 3).

### Table: numeric features by band

**What it shows:** the numeric features cut into bands (birth-year gap in years, address and topic similarity), with pair counts and share of positives; NaN = missing.

**How to read it:** the share of positives should fall as the year gap grows and rise with address similarity.

In [ ]:
bands = {"r_dob_year_gap": [-0.5, 0.5, 1.5, 5.5, 200],
         "r_address_sim": [-0.01, 0.5, 0.8, 0.95, 1.0],
         "x_topics_jaccard": [-0.01, 0.25, 0.5, 0.99, 1.0]}
tabs = []
for c, b in bands.items():
    cut = pd.cut(d[c], b).astype(str).replace("nan", "missing")
    t = d.groupby(cut).agg(pairs=("y", "size"), share_positive=("y", "mean"))
    t.index = pd.MultiIndex.from_product([[c], t.index], names=["feature", "band"])
    tabs.append(t)
pd.concat(tabs).round(3)

**Conclusion.**
- **Year gap falls steeply:** 98.1% positive at gap 0, 11.6% at 1 year, 2.9% at 2–5, 1.3% beyond.
- **Address rises up to 0.95:** from 14.6% positive (similarity ≤ 0.5) to 40.2% (0.5–0.8) and 81.1% (0.8–0.95); above 0.95 slightly lower again (74.6%).
- **Topic overlap is not monotone:** 1.0 = 77.7%, 0.5–0.99 = 2.0%, 0.25–0.5 = 49.2%, ≤ 0.25 = 71.1%. That pattern follows the kinds of lists, not the entities, which fits the decision to leave topics out.

**Next step:** none.

## 3 · Leakage check per feature

For every feature, three numbers, each as AUC and as **signal strength** = max(AUC, 1 − AUC):

- **value AUC:** the feature on its own, computed **only on pairs where the field is present on both sides** (categories replaced by their share of positives). This is what comparing the values achieves.
- **presence AUC:** a 0/1 flag "field present on both sides", on all pairs. This is what the bare existence of the field says.
- **overall AUC:** the feature on all pairs, with "missing" as its own value (NaN for numeric features ranked lowest).

Why signal strength: an AUC below 0.5 means the feature works the other way round (e.g. a distance: low = same entity). That is just as useful to a model, so max(AUC, 1 − AUC) measures how strongly a feature separates the classes, whatever the direction. 0.5 = no signal, 1.0 = perfect. It is **not** a leak measure on its own: a legitimate feature such as an exact birth date is strong too.

**How to read it:** a feature is fine if its **value** signal is strong. It is a leak risk if the **presence** signal is about as strong as the value signal, or stronger: then the field mostly tells *which lists* the records come from (some lists publish birth dates, others not), i.e. the labelling process. Features that share a field (both birth-date features) share the same presence signal.

In [ ]:
def auc_of(x, y):
    # AUC of one feature; categories are replaced by their share of positives
    x, y = pd.Series(x).reset_index(drop=True), pd.Series(y).reset_index(drop=True)
    if str(x.dtype) == "category" or x.dtype == object:
        x = x.astype(str).map(y.groupby(x.astype(str)).mean())
    a = roc_auc_score(y, x.fillna(x.min() - 1))
    return a, max(a, 1 - a)

present = {"r_dob_status": d["r_dob_status"] != "missing", "r_dob_year_gap": d["r_dob_year_gap"].notna(),
           "r_country_status": d["r_country_status"] != "missing", "r_id_status": d["r_id_status"] != "missing",
           "r_address_sim": d["r_address_sim"].notna(), "x_topics_jaccard": d["x_topics_jaccard"].notna()}
rows = []
for c, pres in present.items():
    va, vs = auc_of(d.loc[pres, c], d.loc[pres, "y"])
    pa, ps = auc_of(pres.astype(int), d["y"])
    oa, os_ = auc_of(d[c], d["y"])
    rows.append({"feature": c, "present on both sides": pres.mean(),
                 "value AUC": va, "value strength": vs,
                 "presence AUC": pa, "presence strength": ps,
                 "overall AUC": oa, "overall strength": os_})
pd.DataFrame(rows).set_index("feature").round(3)

**Conclusion.**
- **Value signal clearly beats presence** for birth date (value strength 0.970), year gap (0.948), ID number (0.957) and address (0.788). Presence alone is 0.59–0.72 for them, so these features work by comparing values.
- **Country is the exception:** value strength 0.660, presence strength 0.728. Whether a country is published says more than whether the two countries agree, so it is the most presence-driven feature.
- **Topics:** value strength only 0.599, and excluded anyway (list-based).

**Next step:** test whether country still helps when the missing information is removed (section 5b).

### Why topics stay out

Topic tags (sanction, role.pep, crime.terror, …) are assigned per **list**: every record from a sanctions list gets "sanction", every record from a PEP list "role.pep". Equal topics therefore mostly mean "both records come from the same kind of list", which is the source-list leak from notebook 06 in another form. It describes the publishing process, not the entity, so `x_topics_jaccard` is shown here but not used as a feature.

## 4 · Does it help? Name vs name + record

Split by **person cluster** (as agreed with Vova): IDs linked by positive pairs form a cluster; a cluster goes to validation if a hash of its smallest ID falls below 0.3; a pair is used if both clusters are on the same side. 5 repetitions with different salts. LightGBM with default settings (notebook 07).

Feature sets: **name** (the 15 `n_` features), **record** (the `r_` features only), **name + record**.

In [ ]:
names = pd.read_parquet(C.FEATURES_NAME)
df = pairs[["pair_id", "left_id", "right_id", "y", "person"]].merge(names, on="pair_id").merge(rf, on="pair_id")
df["is_person"] = df["person"].astype(int)

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b in zip(df.loc[df["y"] == 1, "left_id"], df.loc[df["y"] == 1, "right_id"]):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
ents = pd.unique(pd.concat([df["left_id"], df["right_id"]]))
root = pd.Series([find(e) for e in ents], index=ents)
cl_key = root.groupby(root).transform(lambda s: min(s.index))   # smallest ID of the cluster: stable key
df["cl_l"], df["cl_r"] = df["left_id"].map(cl_key), df["right_id"].map(cl_key)

def unit_hash(ids, salt):
    return np.array([int(hashlib.sha1(f"{salt}|{i}".encode()).hexdigest()[:12], 16) / 16**12 for i in ids])
def cluster_side(d_, share, salt):
    cl = pd.unique(pd.concat([d_["cl_l"], d_["cl_r"]]))
    t = pd.Series(unit_hash(cl, salt) < share, index=cl)
    lt, rt = d_["cl_l"].map(t).to_numpy(), d_["cl_r"].map(t).to_numpy()
    return pd.Series(np.where(lt & rt, "test", np.where(~lt & ~rt, "train", "cross")), index=d_.index)

N_COLS = [c for c in df.columns if c.startswith("n_")]
R_COLS = [c for c in df.columns if c.startswith("r_")]
SETS = {"name": N_COLS + ["is_person"], "record": R_COLS + ["is_person"], "name+record": N_COLS + R_COLS + ["is_person"]}
print({k: len(v) for k, v in SETS.items()})

In [ ]:
def at_recall(y, s, recall):
    y, s = np.asarray(y), np.asarray(s)
    order = np.argsort(-s, kind="stable")
    k = min(int(np.searchsorted(np.cumsum(y[order]), np.ceil(recall * y.sum()))), len(y) - 1)
    thr = s[order][k]
    return thr, ((s >= thr) & (y == 0)).sum() / (y == 0).sum()

# evaluation groups: birth date on both sides or not; identical-name negatives
df["dob_group"] = np.where(df["r_dob_status"] != "missing", "birth date both sides", "birth date missing")
df["no_record_info"] = ((df["r_dob_status"] == "missing") & (df["r_country_status"] == "missing")
                        & (df["r_id_status"] == "missing") & df["r_address_sim"].isna())

REPS, rows, scores = 5, [], {}
splits = [(rep, cluster_side(df, 0.3, f"cluster-{rep}")) for rep in range(REPS)]
t0 = time.time()
for rep, side in splits:
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    for name, cols in SETS.items():
        m = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED, verbose=-1)
        s = m.fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
        scores[(rep, name)] = s
        for r in RECALLS:
            thr, fa = at_recall(yv, s, r)
            alarm = s >= thr
            row = {"rep": rep, "set": name, "recall": r, "all": fa, "auc": roc_auc_score(yv, s)}
            for g, mask in [("birth date both sides", va["dob_group"] == "birth date both sides"),
                            ("birth date missing", va["dob_group"] == "birth date missing"),
                            ("identical-name negatives", va["n_exact_name"] == 1),
                            ("no record info", va["no_record_info"])]:
                neg = (yv == 0) & mask.to_numpy()
                row[g] = (alarm & neg).sum() / max(neg.sum(), 1)
            rows.append(row)
res = pd.DataFrame(rows)
print(f"{time.time() - t0:.0f}s; validation pairs per split ~{int(np.mean([(s == 'test').sum() for _, s in splits])):,}")

### Table: false alarms by feature set

**What it shows:** FA rate in % on the validation pairs, mean ± standard deviation over the 5 splits, at each recall level, plus AUC.

**How to read it:** the gap between "name" and "name + record" is what the record features buy. "record" alone shows how far the record fields get without any name comparison.

In [ ]:
t = res.groupby(["set", "recall"])["all"].agg(["mean", "std"]) * 100
t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
t.join(res.groupby("set")["auc"].mean().round(4)).loc[["name", "record", "name+record"]]

**Conclusion.**
- **Name + record cuts false alarms by about 36–46%:** 3.86% → 2.48% at 99% recall (−36%), and 6.34% → 3.45% at 99.5% (−46%).
- **Record features alone are far worse** (25.7%): the names remain the core, the record fields correct them.

**Next step:** find out whether the gain comes from comparing values or from missingness (sections 4 and 5).

### Table: where the gain comes from (99% recall)

**What it shows:** FA rate in % at 99% recall (threshold chosen on all validation pairs), counted inside four groups of negatives: pairs with a birth date on both sides, pairs without, identical-name negatives (n_exact_name = 1), and pairs with **no record information at all** (no birth date, country, ID number or address on at least one side). Last column: in how many of the 5 splits name + record has fewer false alarms than name in that group.

**How to read it:** the gain should sit where record values can be compared (birth date both sides). In the group **"no record info"** the record features carry nothing but the fact that the fields are missing; a clear gain there would be missingness signal, i.e. the leak risk from section 3.

In [ ]:
G = ["all", "birth date both sides", "birth date missing", "identical-name negatives", "no record info"]
t99 = res[res["recall"] == TARGET]
m = t99.groupby("set")[G].mean().loc[["name", "name+record"]] * 100
p = t99.pivot_table(index="rep", columns="set", values=G)
better = pd.Series({g: int((p[(g, "name+record")] < p[(g, "name")]).sum()) for g in G}, name="name+record better (of 5)")
size = pd.Series({"all": (df["y"] == 0).sum(),
                  "birth date both sides": ((df["y"] == 0) & (df["dob_group"] == "birth date both sides")).sum(),
                  "birth date missing": ((df["y"] == 0) & (df["dob_group"] == "birth date missing")).sum(),
                  "identical-name negatives": ((df["y"] == 0) & (df["n_exact_name"] == 1)).sum(),
                  "no record info": ((df["y"] == 0) & df["no_record_info"]).sum()}, name="negatives (all pairs)")
pd.concat([m.round(2), better.to_frame().T, size.to_frame().T])

**Conclusion.** The rows are **groups of pairs**, not features: the model is the same everywhere, only the counting differs. Name + record is better in all groups in 5 of 5 splits.
- **Birth date on both sides:** false alarms fall by 70% (1.64% → 0.50%). That is the clean, value-based gain.
- **Birth date missing:** 4.95% → 3.44%. Country, address and ID number are still compared there, so most of this is legitimate too.
- **No record info at all:** 34.6% → 29.0%. There every record feature is constant, so the drop cannot come from comparing values: it is missingness used as a signal plus the 99%-recall threshold moving because other pairs are separated better.
- **Identical-name negatives:** 67.0% → 49.6%. The floor from notebook 08 shrinks, but stays high.

**Next step:** quantify the missingness part (section 5).

## 5 · Missingness check

Notebook 06 found that which fields are present on both sides predicts the label (AUC 0.79). Two checks here:

1. **Presence-only model:** LightGBM on nothing but four 0/1 flags (birth date / country / ID number / address present on both sides) plus `is_person`. Its AUC is the signal in the missingness pattern alone.
2. **Record features with "missing" removed:** the same name + record model, but with every "missing" category and NaN replaced by one neutral value per feature, so that the model cannot tell missing from "no information". If the false alarms stay about the same, the gain of the record features comes from comparing values, not from missingness.

In [ ]:
df["p_dob"] = (df["r_dob_status"] != "missing").astype(int)
df["p_country"] = (df["r_country_status"] != "missing").astype(int)
df["p_id"] = (df["r_id_status"] != "missing").astype(int)
df["p_address"] = df["r_address_sim"].notna().astype(int)
P_COLS = ["p_dob", "p_country", "p_id", "p_address", "is_person"]

# record features without missingness: "missing" -> the most common non-missing value's neutral stand-in, NaN -> median
neutral = df[R_COLS].copy()
for c in R_CATS:
    neutral[c] = neutral[c].astype(str).replace("missing", "different" if c != "r_dob_status" else "different")
    neutral[c] = pd.Categorical(neutral[c], categories=[x for x in R_CATS[c] if x != "missing"])
for c in ["r_dob_year_gap", "r_address_sim"]:
    neutral[c] = neutral[c].fillna(neutral[c].median())
for c in R_COLS:
    df[c + "_nm"] = neutral[c]
NM_COLS = [c + "_nm" for c in R_COLS]

rows_m = []
for rep, side in splits:
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    for name, cols in {"presence only": P_COLS, "name+record (no missing info)": N_COLS + NM_COLS + ["is_person"]}.items():
        s = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED,
                           verbose=-1).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
        for r in RECALLS:
            rows_m.append({"rep": rep, "set": name, "recall": r, "all": at_recall(yv, s, r)[1],
                           "auc": roc_auc_score(yv, s)})
res_m = pd.concat([res[["rep", "set", "recall", "all", "auc"]], pd.DataFrame(rows_m)])

### Table: missingness check

**What it shows:** FA rate in % (mean over 5 splits) and AUC for: the presence-only model, name, name + record, and name + record with the missing information removed.

**How to read it:** "presence only" shows how much the missingness pattern knows on its own. If "name + record (no missing info)" is about as good as "name + record", the record features earn their gain by comparing values. Note: replacing "missing" by "different" is a deliberately crude stand-in; it removes the information, it is not a model to use.

In [ ]:
t = res_m.groupby(["set", "recall"])["all"].mean().unstack("recall") * 100
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
t = t.join(res_m.groupby("set")["auc"].mean().rename("AUC"))
t.loc[["presence only", "name", "name+record", "name+record (no missing info)"]].round(3)

**Conclusion.**
- **Presence alone is informative but useless for screening:** AUC 0.862, but 82.0% false alarms at 99% recall.
- **Without missing information:** 2.67% instead of 2.48% at 99% recall.
- **Caveat (section 5d): the stand-in is not fully neutral for the birth date.** A missing birth date becomes status "different" with a year gap of 0 (the median). That combination never occurs in real data, since "different" means at least 2 years apart. So the model can still recognise these pairs, and 2.67% still contains some missingness signal.
- The cleaner number is the variant without the year gap (section 5d).

**Next step:** see 5d; as a further check, fill missing values with values drawn at random from pairs where the field is present (section 5e).

## 5b · Without address, without country

Address and country carry much of their signal through their bare presence (section 3). Here the name + record model is trained again without `r_address_sim`, without `r_country_status`, and without both, on the same 5 splits, each **with** and **without** the missing information (stand-in as in section 5).

Question: do the false alarms rise when the two features are left out? If not, they add nothing beyond the presence signal and can be dropped.

In [ ]:
VARIANTS = {
    "name+record": R_COLS,
    "without address": [c for c in R_COLS if c != "r_address_sim"],
    "without country": [c for c in R_COLS if c != "r_country_status"],
    "without address and country": [c for c in R_COLS if c not in ("r_address_sim", "r_country_status")],
}
rows_a = []
t0 = time.time()
for rep, side in splits:
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    no_info = va["no_record_info"].to_numpy()
    for vname, rc in VARIANTS.items():
        for missing_info, cols in [("with", N_COLS + rc + ["is_person"]),
                                   ("without", N_COLS + [c + "_nm" for c in rc] + ["is_person"])]:
            s = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED,
                               verbose=-1).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
            for r in RECALLS:
                thr, fa = at_recall(yv, s, r)
                neg_ni = (yv == 0) & no_info
                rows_a.append({"rep": rep, "variant": vname, "missing info": missing_info, "recall": r, "fa": fa,
                               "fa no record info": ((s >= thr) & neg_ni).sum() / max(neg_ni.sum(), 1),
                               "auc": roc_auc_score(yv, s)})
res_a = pd.DataFrame(rows_a)
print(f"{time.time() - t0:.0f}s")

### Table: false alarms per variant

**What it shows:** FA rate in %, mean ± standard deviation over the 5 splits, for each variant, with and without the missing information. Last column: FA rate at 99% recall inside the pairs with no record information at all (where any gain can only come from missingness).

**How to read it:** compare each "without …" row with the full "name+record" row of the same missing-info setting. A rise smaller than the standard deviation means the feature adds nothing. A falling value in the last column means the variant uses less missingness signal.

In [ ]:
t = res_a.groupby(["missing info", "variant", "recall"])["fa"].agg(["mean", "std"]) * 100
t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
ni = res_a[res_a["recall"] == TARGET].groupby(["missing info", "variant"])["fa no record info"].mean() * 100
t = t.join(ni.round(2).rename("99%: no record info")).join(res_a.groupby(["missing info", "variant"])["auc"].mean().round(4))
t.reindex(pd.MultiIndex.from_product([["with", "without"], list(VARIANTS)]))

**Conclusion.** Leaving out address or country raises the false alarms at every recall level, with and without the missing information. Leaving out country also raises the false alarms in the pairs without record info (its missingness is used); for address that column hardly moves (29.0% → 28.9% with missing info).

**Next step:** see the paired table below for whether the rise is consistent.

### Table: paired comparison at 99% recall

**What it shows:** for each variant, the change in FA rate (pp) against the full name + record model of the same missing-info setting, mean over the splits, and in how many of the 5 splits the variant has **more** false alarms.

**How to read it:** a change near 0 with "worse in" at 3 or fewer of 5 splits means the left-out feature is not needed.

In [ ]:
a = res_a[res_a["recall"] == TARGET].pivot_table(index="rep", columns=["missing info", "variant"], values="fa")
rows = []
for mi in ["with", "without"]:
    base = a[(mi, "name+record")]
    for v in list(VARIANTS)[1:]:
        diff = (a[(mi, v)] - base) * 100
        rows.append({"missing info": mi, "variant": v, "change at 99% (pp)": round(diff.mean(), 3),
                     "worse in (of 5)": int((diff > 0).sum())})
pd.DataFrame(rows).set_index(["missing info", "variant"])

**Conclusion.**
- **Both features help consistently:** without address +0.10 / +0.26 pp at 99% recall (with / without missing info), without country +0.23 / +0.09 pp, both worse in 5 of 5 splits. **Keep both.**
- **For country** a larger part of the effect comes from missingness (+0.23 → +0.09 pp once the missing information is removed), as section 3 suggested. Address works through its values (the effect even grows without missing info).

**Next step:** none for the selection; report the country feature with this caveat.

### Findings: section 5b

**Address and country both help, also without the missing information.** At 99% recall the false alarms rise in 5 of 5 splits when either is left out:

| Left out | with missing info | without missing info |
| --- | --- | --- |
| address | +0.10 pp | +0.26 pp |
| country | +0.23 pp | +0.09 pp |
| both | +0.40 pp | +0.35 pp |

- So both carry signal from comparing values, not only from their presence. Country's value effect is small (+0.09 pp).
- **Decision: keep both.**

**Caveat on the missingness checks.** In the pairs without any record information, false alarms fall from 34.6% (name only) to 29.7% even with the missing information removed. So that group does not isolate the missingness effect as cleanly as assumed in section 4: when the record features separate other pairs better, the threshold for 99% recall moves, and this also changes the alarms in this group.
- The stand-in ("missing" → "different") is not neutral either: it marks those pairs as "different", which lowers their scores.
- The cleanest evidence that the values work is the gain **inside the pairs where the field is present on both sides** (birth date: 1.64% → 0.50%).
- First estimate of the missingness part: **about 0.2 pp** (2.48% → 2.67%) of the 1.38 pp gain; section 5e shows it is larger.

## 5c · Birth date, version 2: day, month, swaps and digit typos

Version 1 (`r_dob_status`) is built around the year: a typo in the year ("1968" vs "1986") or day and month swapped (US vs European order) ends up in "different" or "same_year". Version 2 (`r_dob_status_v2`) adds these cases. Categories, in order (the best relation over all birth dates of both sides counts):

| Category | Meaning | Example |
| --- | --- | --- |
| exact | same full date | 1968-03-02 / 1968-03-02 |
| partial_match | equal as far as both are known | 1968 / 1968-03-02 |
| swapped | same year, day and month swapped | 1968-03-02 / 1968-02-03 |
| digit_typo | exactly one of year / month / day differs, by one changed digit or two neighbouring digits swapped | 1968 / 1986, 1968-03-02 / 1968-08-02, 1968-03-12 / 1968-03-21 |
| same_year_month | same year and month, other day | 1968-03-02 / 1968-03-17 |
| same_day_month | same day and month, year differs (not a digit typo) | 1968-03-02 / 1975-03-02 |
| same_year | only the year matches | 1968-03-02 / 1968-11-20 |
| close | years 1 apart | 1967-03-02 / 1968-11-20 |
| different | everything else | |
| missing | no birth date on at least one side | |

A year that differs by one (1968 / 1969) is a one-digit change too; it counts as **digit_typo** when month and day agree, otherwise as **close**. The extra column `typo_part` (day, month, year (digit), year (swap)) is only for the analysis, not a feature.

**Version 3** (`r_dob_status_v3`) is the same, except that a single changed digit in the **year** is not a typo (it then falls into same_day_month, close or different); swapped year digits (1968 / 1986) and typos in day or month still are. Reason: a first run showed that one changed digit in the year occurs mostly between different people (two years up to 9 apart).

In [ ]:
WIDTH = (4, 2, 2)
def one_edit(x, y, width):
    """'digit' (one digit changed), 'swap' (two neighbouring digits swapped) or None."""
    a, b = f"{x:0{width}d}", f"{y:0{width}d}"
    diff = [i for i in range(width) if a[i] != b[i]]
    if len(diff) == 1:
        return "digit"
    if len(diff) == 2 and diff[1] == diff[0] + 1 and a[diff[0]] == b[diff[1]] and a[diff[1]] == b[diff[0]]:
        return "swap"
    return None

DOB_ORDER_V2 = ["exact", "partial_match", "swapped", "digit_typo", "same_year_month", "same_day_month",
                "same_year", "close", "different"]
PART = ("year", "month", "day")
def dob_relation_v2(a, b, year_digit=True):
    """year_digit=False (version 3): a single changed digit in the year does not count as a typo."""
    if len(a) == 3 and a == b:
        return "exact", None
    k = min(len(a), len(b))
    if a[:k] == b[:k]:
        return "partial_match", None
    if k == 3 and a[0] == b[0] and a[1] == b[2] and a[2] == b[1]:
        return "swapped", None
    diff = [i for i in range(k) if a[i] != b[i]]
    if len(diff) == 1 and not (k == 1 and abs(a[0] - b[0]) == 1):
        kind = one_edit(a[diff[0]], b[diff[0]], WIDTH[diff[0]])
        if kind and (diff[0] > 0 or kind == "swap" or year_digit):
            return "digit_typo", PART[diff[0]] + (f" ({kind})" if diff[0] == 0 else "")
    if k == 3 and a[:2] == b[:2]:
        return "same_year_month", None
    if k == 3 and a[1:] == b[1:]:
        return "same_day_month", None
    if a[0] == b[0]:
        return "same_year", None
    if abs(a[0] - b[0]) == 1:
        return "close", None
    return "different", None

def dob_v2(r, year_digit=True):
    L = [x for x in map(parse_dob, values(r, "left", ["birthDate"])) if x]
    R = [x for x in map(parse_dob, values(r, "right", ["birthDate"])) if x]
    if not L or not R:
        return "missing", None
    rels = [dob_relation_v2(a, b, year_digit) for a in L for b in R]
    best = min(rels, key=lambda t: DOB_ORDER_V2.index(t[0]))
    return best

# checks on hand-made dates
for a, b in [((1968, 3, 2), (1968, 2, 3)), ((1968,), (1986,)), ((1968, 3, 2), (1968, 8, 2)), ((1968, 3, 12), (1968, 3, 21)),
             ((1968, 3, 2), (1975, 3, 2)), ((1968, 3, 2), (1969, 3, 2)), ((1968,), (1969,)), ((1968, 3, 2), (1968, 3, 17))]:
    print(a, b, dob_relation_v2(a, b))

t0 = time.time()
v2 = [dob_v2(r) for r in pairs.itertuples(index=False)]
v3 = [dob_v2(r, year_digit=False) for r in pairs.itertuples(index=False)]
cats = DOB_ORDER_V2 + ["missing"]
dob2 = pd.DataFrame({"pair_id": pairs["pair_id"],
                     "r_dob_status_v2": pd.Categorical([x[0] for x in v2], categories=cats),
                     "typo_part": [x[1] for x in v2],
                     "r_dob_status_v3": pd.Categorical([x[0] for x in v3], categories=cats)})
df = df.drop(columns=[c for c in dob2.columns if c in df.columns and c != "pair_id"]).merge(dob2, on="pair_id")
print(f"{time.time() - t0:.0f}s")

### Table: version 1 against version 2

**What it shows:** how the pairs of each version-1 category (rows) are distributed over the version-2 categories (columns). Only categories that change are interesting: "exact", "partial_match" and "missing" stay as they are.

**How to read it:** the cells outside the old categories show which pairs version 2 sorts differently, e.g. how many former "different" pairs are now "digit_typo" or "same_day_month".

In [ ]:
pd.crosstab(df["r_dob_status"], df["r_dob_status_v2"]).loc[:, lambda t: t.sum() > 0]

**Conclusion.** Version 2 reshuffles mainly three old categories.
- **"different":** 4,406 of its 18,413 pairs become "digit_typo", almost all through a one-digit change in the year; 70 become "same_day_month".
- **"same_year":** its 1,054 pairs split into swapped (84), typo (498), same year and month (68) and same year only (404).
- **"close":** 233 of 1,393 become "digit_typo".
- "exact", "partial_match" and "missing" stay unchanged.

**Next step:** check whether the new categories differ in their share of positives (next table).

### Table: version 2 categories and share of positives

**What it shows:** number of pairs, share of positive pairs and number of positive pairs per version-2 category; for "digit_typo" also split by the part that differs (year / month / day).

**How to read it:** a new category is useful if its share of positives is clearly different from "different" (1.7%): high values (e.g. for "swapped" or "digit_typo") mean these are mostly the same person with a typo.

In [ ]:
t = df.groupby("r_dob_status_v2", observed=False).agg(pairs=("y", "size"), share_positive=("y", "mean"), positives=("y", "sum"))
tp = (df[df["r_dob_status_v2"] == "digit_typo"].groupby("typo_part")
      .agg(pairs=("y", "size"), share_positive=("y", "mean"), positives=("y", "sum")))
tp.index = "  digit_typo: " + tp.index
pd.concat([t, tp]).round(3)

**Conclusion.**
- **Real typos are rare:** day/month swapped (97.6% positive), a typo in the day (96.3%) and a typo in the month (95.2%) are almost always the same person, but together only about 570 pairs.
- **Typos in the year are not typos:** a changed digit (4.2%) or swapped digits (2.7%) mostly mean different people with similar years.

**Next step:** if anything, a version 4 with only swapped day/month and day/month typos; useful for the error taxonomy rather than for the model.

### Table: what happens to the positive pairs with a "different" birth date

**What it shows:** the positive pairs that version 1 calls "different" (the hardest positives for the birth-date feature), and the version-2 category they end up in.

**How to read it:** every pair that moves out of "different" into a typo-like category is a true match the model no longer has to push down.

In [ ]:
pos_diff = df[(df["y"] == 1) & (df["r_dob_status"] == "different")]
t = pos_diff["r_dob_status_v2"].value_counts().rename("positive pairs").to_frame()
t["share"] = t["positive pairs"] / len(pos_diff)
print(f"positive pairs with version-1 'different': {len(pos_diff):,}")
t[t["positive pairs"] > 0].round(3)

**Conclusion.** Of the 315 positive pairs with a "different" birth date, 129 move to "digit_typo" (year changes, which the model cannot trust), 17 to "same_day_month" and 169 stay "different". Typos explain few of these hard positives.

**Next step:** none. These pairs stay in the error taxonomy as a group of their own: hard true matches with conflicting birth dates (a wrong birth date in one list, or a label error; the data cannot tell the two apart).

### Model comparison: version 1 vs version 2

Same 5 splits, same LightGBM, name + record features (with missing information, as in section 4); the only change is `r_dob_status` → `r_dob_status_v2` or `r_dob_status_v3`.

Two tables follow:

1. **FA rate** in % at each recall (mean ± standard deviation) and in how many of the 5 splits version 2 has fewer false alarms at 99% and 99.5% recall.
2. **The hard positives:** of the positive validation pairs that version 1 calls "different", how many lie below the threshold for 99% recall (= missed) with each version, summed over the 5 splits.

**How to read it:** if a new version helps, the misses among these positives drop, and the threshold can stay higher, which shows as fewer false alarms at 99–99.5% recall. The gain will be small because these cases are rare; what matters is whether it appears in most splits.

In [ ]:
COLS_V1 = N_COLS + R_COLS + ["is_person"]
COLS_V2 = N_COLS + [c if c != "r_dob_status" else "r_dob_status_v2" for c in R_COLS] + ["is_person"]
COLS_V3 = N_COLS + [c if c != "r_dob_status" else "r_dob_status_v3" for c in R_COLS] + ["is_person"]
rows_v, miss = [], []
for rep, side in splits:
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    hard = ((va["y"] == 1) & (va["r_dob_status"] == "different")).to_numpy()
    for ver, cols in [("version 1", COLS_V1), ("version 2", COLS_V2), ("version 3", COLS_V3)]:
        s = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED,
                           verbose=-1).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
        for r in RECALLS:
            thr, fa = at_recall(yv, s, r)
            rows_v.append({"rep": rep, "version": ver, "recall": r, "fa": fa})
            if r == TARGET:
                miss.append({"rep": rep, "version": ver, "hard positives": int(hard.sum()),
                             "missed": int((hard & (s < thr)).sum())})
res_v = pd.DataFrame(rows_v)
t = res_v.groupby(["version", "recall"])["fa"].agg(["mean", "std"]) * 100
t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
pv = res_v.pivot_table(index="rep", columns=["recall", "version"], values="fa")
for v in ("version 2", "version 3"):
    t[f"{v} better than version 1 at 99% / 99.5% (of 5)"] = ""
    t.loc[v, f"{v} better than version 1 at 99% / 99.5% (of 5)"] = " / ".join(
        str(int((pv[(r, v)] < pv[(r, "version 1")]).sum())) for r in (0.99, 0.995))
t

**Conclusion.** No version beats version 1 consistently: 2.48% / 2.46% / 2.44% at 99% recall and 3.45% / 3.44% / 3.42% at 99.5%, with differences far below the standard deviation; versions 2 and 3 are better in only 2–4 of 5 splits.

**Next step:** test a slimmer version 4 with only the real typos (section 5f).

In [ ]:
m = pd.DataFrame(miss).groupby("version")[["hard positives", "missed"]].sum()
m["share missed"] = (m["missed"] / m["hard positives"]).round(3)
m

**Conclusion.** The new versions do not find more of the hard positives: version 1 misses 132 of 451 over the 5 splits, version 2 138, version 3 130.

**Next step:** none.

### Findings: section 5c

**Typos in day or month and swapped day/month are real, but rare.** These pairs are almost always the same person; version 1 had them in "same_year" (69% positive) or "close".

| Category | Pairs | Positive |
| --- | --- | --- |
| day and month swapped | 84 | 98% |
| one digit wrong / swapped in the day | 296 | 96% |
| one digit wrong / swapped in the month | 187 | 95% |

**Typos in the year are not typos.** One changed digit in the year (4,504 pairs) is 4.2% positive, swapped year digits such as 1968 / 1986 (150 pairs) 2.7%. These are almost always different people whose years happen to look alike. Same day and month with another year: 28.9% positive (76 pairs).

**Positive pairs that version 1 calls "different":** 315.
- 129 of them move to digit_typo, all year changes, which the model cannot trust (about 4% positive).
- 17 move to same_day_month.
- 169 stay "different".
- So the hard positives are mostly not explained by typos; they are more likely wrong birth dates in one of the lists or label errors.

**No measurable gain in the model.** At 99% recall:

| Version | false alarms | better than version 1 (of 5 splits, 99% / 99.5%) |
| --- | --- | --- |
| version 1 | 2.48% | |
| version 2 | 2.46% | 2 / 4 |
| version 3 | 2.44% | 3 / 2 |

- At 99.5% recall: 3.45%, 3.44%, 3.42%.
- Missed hard positives: 132 (version 1), 138 (version 2), 130 (version 3) of 451 over the 5 splits.

**Recommendation:** versions 2 and 3 bring no measurable gain. A simpler version 4 that adds only `swapped` and a typo in day or month (no year typos) is tested in section 5f.

## 5d · Is the birth-year gap needed?

`r_dob_year_gap` repeats most of what `r_dob_status` already says (exact / partial / same year / close / different); it only adds how far apart the years are. Its presence signal is identical to that of `r_dob_status` (both are missing exactly when the birth date is missing), so it adds no missingness information either.

Test: name + record with and without `r_dob_year_gap`, same 5 splits, with and without the missing information (as in section 5b).

**What the table shows:** FA rate in % (mean ± standard deviation over the 5 splits) at each recall, and in how many of the 5 splits the model **without** the year gap has **more** false alarms at 99% / 99.5% recall.

**How to read it:** if leaving it out does not raise the false alarms in most splits (4 or 5 of 5), the year gap is not needed and can be dropped: one feature fewer to explain.

In [ ]:
rows_g = []
for rep, side in splits:
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    for gap_in in ("with year gap", "without year gap"):
        rc = R_COLS if gap_in == "with year gap" else [c for c in R_COLS if c != "r_dob_year_gap"]
        for mi, cols in [("with", N_COLS + rc + ["is_person"]), ("without", N_COLS + [c + "_nm" for c in rc] + ["is_person"])]:
            s = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED,
                               verbose=-1).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
            for r in RECALLS:
                rows_g.append({"rep": rep, "missing info": mi, "variant": gap_in, "recall": r, "fa": at_recall(yv, s, r)[1]})
res_g = pd.DataFrame(rows_g)
t = res_g.groupby(["missing info", "variant", "recall"])["fa"].agg(["mean", "std"]) * 100
t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
pv = res_g.pivot_table(index="rep", columns=["missing info", "variant", "recall"], values="fa")
t["without worse at 99% / 99.5% (of 5)"] = ""
for mi in ("with", "without"):
    t.loc[(mi, "without year gap"), "without worse at 99% / 99.5% (of 5)"] = " / ".join(
        str(int((pv[(mi, "without year gap", r)] > pv[(mi, "with year gap", r)]).sum())) for r in (0.99, 0.995))
t

**Conclusion.**
- **With the missing information (the normal model):** leaving out the year gap changes nothing. At 99% recall 2.48% → 2.48%, worse in only 1 of 5 splits; at 99.5% recall 3.45% → 3.43%, worse in 3 of 5.
- **Without the missing information it seems to matter** (2.67% → 2.73%, 4 of 5). But that is the stand-in flaw from section 5: with the year gap, a missing birth date is recognisable ("different" + gap 0). Without it, missing simply looks like "different".
- **So the year gap adds no real value.** And **2.73% is the cleaner estimate of the model without missingness signal.**

**Next step:** drop `r_dob_year_gap`. Main result: name + record without the year gap. Report it both with missing information (2.48%) and without (2.73%).

### Findings: section 5d

- **Drop the birth-year gap.** In the normal model it adds nothing measurable: 2.48% → 2.48% at 99% recall, worse in only 1 of 5 splits.
- Its apparent value without missing information is an artefact of the stand-in: "different" + gap 0 marks a missing birth date.
- **Cleaner estimate of the missingness part:** without year gap, name + record reaches 2.48% with and 2.73% without missing information.
- Of the gain over names only (3.86%), about **1.1 pp comes from comparing values** (3.86% → 2.73%) and about **0.25 pp from missingness** (2.73% → 2.48%).

## 5e · Missingness check with a random stand-in

Sections 5 and 5d remove the missing information by replacing "missing" with "different" (NaN with the median). That stand-in is not neutral: it marks every pair with a missing field as "different", which pushes it towards "not the same entity".

Cleaner check: fill every missing value with the value of a **randomly drawn pair where that field is present** (drawn from all such pairs, independently of the label). The filled-in value then carries no information about the label, and the model cannot tell "missing" from a real value; it only sees noise. Each of the 5 splits uses its own random draw.

All variants without the year gap (section 5d):

| Variant | Missing values |
| --- | --- |
| name only | (no record features) |
| with missing info | kept as "missing" / NaN |
| stand-in "different" | "missing" → "different", NaN → median (as in section 5) |
| stand-in random | value of a random pair where the field is present |

In [ ]:
R_MAIN = [c for c in R_COLS if c != "r_dob_year_gap"]
MISSING = {"r_dob_status": (df["r_dob_status"] == "missing").to_numpy(),
           "r_country_status": (df["r_country_status"] == "missing").to_numpy(),
           "r_id_status": (df["r_id_status"] == "missing").to_numpy(),
           "r_address_sim": df["r_address_sim"].isna().to_numpy()}

def random_fill(seed):
    # fill missing values with values drawn at random from pairs where the field is present
    rng = np.random.default_rng(seed)
    out = {}
    for c, m in MISSING.items():
        vals = df[c].to_numpy(dtype=object if c in R_CATS else float)
        present = vals[~m]
        filled = vals.copy()
        filled[m] = present[rng.integers(0, len(present), m.sum())]
        if c in R_CATS:
            out[c + "_rnd"] = pd.Categorical(filled, categories=[x for x in R_CATS[c] if x != "missing"])
        else:
            out[c + "_rnd"] = filled.astype(float)
    return pd.DataFrame(out, index=df.index)

VARIANTS_M = {"name only": N_COLS + ["is_person"],
              "with missing info": N_COLS + R_MAIN + ["is_person"],
              "stand-in different": N_COLS + [c + "_nm" for c in R_MAIN] + ["is_person"],
              "stand-in random": N_COLS + [c + "_rnd" for c in R_MAIN] + ["is_person"]}
rows_r = []
t0 = time.time()
for rep, side in splits:
    rnd = random_fill(C.RANDOM_SEED + rep)
    for c in rnd.columns:
        df[c] = rnd[c]
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    no_info = va["no_record_info"].to_numpy()
    for vname, cols in VARIANTS_M.items():
        s = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED,
                           verbose=-1).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
        for r in RECALLS:
            thr, fa = at_recall(yv, s, r)
            neg_ni = (yv == 0) & no_info
            rows_r.append({"rep": rep, "variant": vname, "recall": r, "fa": fa,
                           "fa no record info": ((s >= thr) & neg_ni).sum() / max(neg_ni.sum(), 1),
                           "auc": roc_auc_score(yv, s)})
res_r = pd.DataFrame(rows_r)
print(f"{time.time() - t0:.0f}s")

### Table: false alarms with the three ways of handling missing values

**What it shows:** FA rate in % (mean ± standard deviation over the 5 splits) at each recall, the FA rate at 99% recall inside the pairs with no record information at all, and the AUC.

**How to read it:**
- **The gap between "with missing info" and "stand-in random"** is what the model gains from missingness alone.
- **The gap between "name only" and "stand-in random"** is what the record features gain by comparing values.
- **If "stand-in random" is close to "stand-in different",** the earlier estimate from sections 5 and 5d holds.
- **The no-record-info column:** with a random stand-in, the record features carry only noise in that group, so any remaining drop against "name only" comes from the threshold moving, not from missingness.

In [ ]:
order = list(VARIANTS_M)
t = res_r.groupby(["variant", "recall"])["fa"].agg(["mean", "std"]) * 100
t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
ni = res_r[res_r["recall"] == TARGET].groupby("variant")["fa no record info"].mean() * 100
t = t.join(ni.round(2).rename("99%: no record info")).join(res_r.groupby("variant")["auc"].mean().round(4))
t.loc[order]

**Conclusion.**
- **With the random stand-in, the gain over names alone shrinks to 0.56 pp** at 99% recall (3.86% → 3.30%). With the "different" stand-in it was 1.13 pp (2.73%), with the missing information 1.38 pp (2.48%).
- **In the pairs without any record information** the random stand-in is almost back at the name-only level (33.1% vs 34.6%), so it removes most of the missingness signal.
- **But the random stand-in is too pessimistic.** About half of all pairs get a random value, mostly a fake "exact" birth date (most present pairs match), sometimes a fake "different". The model can no longer tell real from fake values, so it has to trust every birth date less, also the real ones. That costs part of the real value signal too.

**So the value-based gain lies between 0.56 pp (random stand-in, a lower bound) and 1.13 pp ("different" stand-in, still slightly optimistic).**

**Next step:** none; report the range (see the findings below).

### Table: paired comparison at 99% and 99.5% recall

**What it shows:** for each pair of variants, the mean difference in FA rate (pp, row minus column) and in how many of the 5 splits the row variant has **fewer** false alarms.

**How to read it:** a difference with "fewer in" at 4 or 5 of 5 is consistent; 2–3 of 5 is noise.

In [ ]:
pv = res_r.pivot_table(index="rep", columns=["recall", "variant"], values="fa")
rows = []
for r in (0.99, 0.995):
    for a, b in [("stand-in random", "name only"), ("with missing info", "stand-in random"),
                 ("stand-in random", "stand-in different")]:
        diff = (pv[(r, a)] - pv[(r, b)]) * 100
        rows.append({"recall": f"{r:.1%}", "comparison": f"{a} vs {b}", "difference (pp)": round(diff.mean(), 3),
                     "first has fewer (of 5)": int((diff < 0).sum())})
pd.DataFrame(rows).set_index(["recall", "comparison"])

**Conclusion.**
- **All three gaps are consistent in 5 of 5 splits:** the random stand-in beats names only, the missing information beats the random stand-in, and the "different" stand-in beats the random stand-in.
- **The order of the variants is therefore certain;** only the size of the value-based gain is uncertain (0.56–1.13 pp at 99% recall, 1.02–2.45 pp at 99.5%).

### Findings: section 5e

- **Names only: 3.86%** false alarms at 99% recall.

| Variant | 99% recall | gain over names only |
| --- | --- | --- |
| with missing information | 2.48% | 1.38 pp |
| "different" stand-in | 2.73% | 1.13 pp |
| random stand-in | 3.30% | 0.56 pp |

- **Missingness is a real part of the record-feature gain:** between about 0.25 pp ("different" stand-in) and 0.8 pp (random stand-in) of the 1.38 pp.
- **Neither stand-in is clean.** "different" pushes pairs with missing fields towards "not the same". The random value adds noise that also weakens the real values. The truth lies in between.
- **Reweighting** (sample weights that make the presence pattern independent of the label) would give a cleaner number, but we decided against it: too complex for what it adds.
- **How to report it:** as a range. Names + record cut false alarms at 99% recall by 0.6–1.1 pp through comparing values (15–29%); everything beyond that up to 1.38 pp (2.48%) comes from the missingness pattern, i.e. from how the lists were built. Main result: the "different" stand-in (2.73%).

## 5f · Birth date, version 4: only the real typos

Section 5c showed that only two of the new birth-date cases are real typos: **day and month swapped** and **one digit wrong or swapped in the day or month** (95–98% positive). Year "typos" and "same day and month, other year" are mostly different people. Version 4 (`r_dob_status_v4`) therefore adds only these two categories to version 1; everything else is decided exactly as in version 1.

| Category | Meaning | Example |
| --- | --- | --- |
| exact, partial_match | as in version 1 | |
| swapped | same year, day and month swapped | 1968-03-02 / 1968-02-03 |
| day_month_typo | same year, and day **or** month differs by one changed digit or two swapped digits | 1968-03-12 / 1968-03-21, 1968-03-02 / 1968-08-02 |
| same_year, close, different, missing | as in version 1 | |

Test: the model as it will go into the code, i.e. name + record **without the year gap** (section 5d), with version 1 or version 4, on the same 5 splits, with the missing information and with the "different" stand-in (the main result, section 5e).

In [ ]:
DOB_ORDER_V4 = ["exact", "partial_match", "swapped", "day_month_typo", "same_year", "close", "different"]
def dob_relation_v4(a, b):
    rel, part = dob_relation_v2(a, b, year_digit=False)
    if rel in ("exact", "partial_match", "swapped"):
        return rel
    if rel == "digit_typo" and part in ("month", "day"):
        return "day_month_typo"
    return dob_relation(a, b)            # everything else exactly as version 1

def dob_v4(r):
    L = [x for x in map(parse_dob, values(r, "left", ["birthDate"])) if x]
    R = [x for x in map(parse_dob, values(r, "right", ["birthDate"])) if x]
    if not L or not R:
        return "missing"
    rels = {dob_relation_v4(a, b) for a in L for b in R}
    return next(x for x in DOB_ORDER_V4 if x in rels)

# checks on hand-made dates
for a, b in [((1968, 3, 2), (1968, 2, 3)), ((1968, 3, 12), (1968, 3, 21)), ((1968, 3, 2), (1968, 8, 2)),
             ((1968,), (1986,)), ((1968, 3, 2), (1969, 3, 2)), ((1968, 3, 2), (1975, 3, 2)), ((1968, 3, 2), (1968, 11, 20))]:
    print(a, b, dob_relation_v4(a, b))

t0 = time.time()
v4 = dict(zip(pairs["pair_id"], (dob_v4(r) for r in pairs.itertuples(index=False))))
df["r_dob_status_v4"] = pd.Categorical(df["pair_id"].map(v4), categories=DOB_ORDER_V4 + ["missing"])
assert df["r_dob_status_v4"].notna().all()
assert ((df["r_dob_status_v4"] == "missing") == (df["r_dob_status"] == "missing")).all()
df["r_dob_status_v4_nm"] = pd.Categorical(df["r_dob_status_v4"].astype(str).replace("missing", "different"),
                                          categories=DOB_ORDER_V4)
print(f"{time.time() - t0:.0f}s")

### Table: version 4 against version 1

**What it shows:** for each version-4 category, the number of pairs, the share of positive pairs, and the version-1 category these pairs came from.

**How to read it:** "swapped" and "day_month_typo" should be clearly positive and come out of version 1's "same_year" (often negative), so that the model no longer has to push these true matches down. All other categories must be unchanged.

In [ ]:
t = pd.crosstab(df["r_dob_status_v4"], df["r_dob_status"])
t = t.loc[:, t.sum() > 0]
t.insert(0, "share positive", df.groupby("r_dob_status_v4", observed=False)["y"].mean().round(3))
t.insert(0, "pairs", df["r_dob_status_v4"].value_counts())
t

**Conclusion.**
- **Only "same_year" changes:** of its 1,054 pairs, 84 become "swapped" (97.6% positive) and 486 "day_month_typo" (95.3%). The remaining 484 "same_year" pairs are now only 37.2% positive instead of 68.8%, so "same_year" becomes a much cleaner signal too.
- All other categories are exactly as in version 1, as intended.

**Next step:** check whether the model gains from it (next table).

### Table: false alarms with version 1 and version 4

**What it shows:** FA rate in % (mean ± standard deviation over the 5 splits) at each recall, for version 1 and version 4, with the missing information and with the "different" stand-in; in how many of the 5 splits version 4 has **fewer** false alarms at 99% / 99.5% recall; and the positive validation pairs that version 4 moves into "swapped" or "day_month_typo": how many of them each version misses at 99% recall (summed over the 5 splits).

**How to read it:** version 4 is worth keeping if it is not worse in most splits and misses fewer of the moved positives. A gain in the overall FA rate is unlikely, because the new categories cover only a few hundred pairs.

In [ ]:
rows_4, miss_4 = [], []
R_V4 = ["r_dob_status_v4" if c == "r_dob_status" else c for c in R_MAIN]
t0 = time.time()
for rep, side in splits:
    tr, va = df[side == "train"], df[side == "test"]
    yv = va["y"].to_numpy()
    moved = ((va["y"] == 1) & va["r_dob_status_v4"].isin(["swapped", "day_month_typo"])).to_numpy()
    for mi in ("with missing info", "stand-in different"):
        for ver, rc in [("version 1", R_MAIN), ("version 4", R_V4)]:
            cols = N_COLS + (rc if mi == "with missing info" else [c + "_nm" for c in rc]) + ["is_person"]
            s = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=C.RANDOM_SEED,
                               verbose=-1).fit(tr[cols], tr["y"]).predict_proba(va[cols])[:, 1]
            for r in RECALLS:
                thr, fa = at_recall(yv, s, r)
                rows_4.append({"rep": rep, "missing info": mi, "version": ver, "recall": r, "fa": fa})
                if r == TARGET:
                    miss_4.append({"missing info": mi, "version": ver, "moved positives": int(moved.sum()),
                                   "missed": int((moved & (s < thr)).sum())})
res_4 = pd.DataFrame(rows_4)
print(f"{time.time() - t0:.0f}s")

t = res_4.groupby(["missing info", "version", "recall"])["fa"].agg(["mean", "std"]) * 100
t = t.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1).unstack("recall")
t.columns = [f"FA % at {c:.1%}" for c in t.columns]
pv = res_4.pivot_table(index="rep", columns=["missing info", "version", "recall"], values="fa")
t["version 4 better at 99% / 99.5% (of 5)"] = ""
for mi in ("with missing info", "stand-in different"):
    t.loc[(mi, "version 4"), "version 4 better at 99% / 99.5% (of 5)"] = " / ".join(
        str(int((pv[(mi, "version 4", r)] < pv[(mi, "version 1", r)]).sum())) for r in (0.99, 0.995))
m = pd.DataFrame(miss_4).groupby(["missing info", "version"])[["moved positives", "missed"]].sum()
t.join(m)

**Conclusion.**
- **The overall FA rate does not change measurably:** 2.73% → 2.71% at 99% recall with the "different" stand-in, 2.48% → 2.47% with missing information; differences far below the standard deviation, version 4 better in 2–4 of 5 splits.
- **But version 4 finds the true matches it was built for:** of the 842 moved positive pairs (summed over the 5 splits), version 1 misses 12 (stand-in) / 16 (with missing info), version 4 only 4 / 2.
- So version 4 costs nothing in false alarms and recovers real matches with a typo or swapped date, which is exactly what a screening system should not miss.

**Decision:** use version 4 as `r_dob_status` in `src/screening/record_features.py`.

### Findings: section 5f

- **Version 4** = version 1 plus `swapped` (84 pairs, 98% positive) and `day_month_typo` (486 pairs, 95% positive); both come out of "same_year", which becomes cleaner (69% → 37% positive).
- **False alarms unchanged** (2.73% → 2.71% at 99% recall, "different" stand-in; within noise).
- **Fewer missed true matches:** of the moved positives, 4 instead of 12 missed (stand-in) and 2 instead of 16 (with missing info), over the 5 splits.
- **Decision: version 4 replaces version 1.** It is easy to explain (two real typo patterns, e.g. US vs European date order) and costs nothing.

## 6 · Output contract (for the move into code)

When moved to `src/screening/record_features.py`, the output `features_record.parquet` has `pair_id` + the `r_` columns, same rows as `PAIRS_INPUT`:

| Column | Type | Categories / range |
| --- | --- | --- |
| r_dob_status | category | exact, partial_match, same_year, close, different, missing |
| r_dob_year_gap | float, NaN if missing | ≥ 0 |
| r_country_status | category | same, different, missing |
| r_id_status | category | same, different, missing |
| r_address_sim | float, NaN if missing | 0–1 |

Notebook 07 picks the file up automatically (feature set `name+record`).

In [ ]:
out = rf[["pair_id"] + R_COLS]
assert len(out) == len(pairs) and out["pair_id"].is_unique
assert all(out[c].notna().all() for c in R_CATS), "categorical columns must never be NaN"
print(out.dtypes.to_string())
print(f"{len(out):,} rows; not written to disk (notebooks explore, scripts produce)")

## Findings (cleaned data, Vova's train + val, split by person cluster, LightGBM, 2026-11-25)

**The values carry strong signal.**
- Birth date: exact match 99.0% positive, different 1.7%.
- ID number: same 99.1% positive, different 1.3%.
- Country: different 4.0% positive; "same country" says little (68.2% vs 74.5% overall; many confusions are within one country).
- Address: rises from 15% positive (similarity ≤ 0.5) to 81% (0.8–0.95).

**Record features cut the false alarms by about 36–46%** (all five record features; without the year gap, see 5d, 2.48% / 3.43%).

| | 99% recall | 99.5% recall |
| --- | --- | --- |
| name only | 3.86% | 6.34% |
| name + record | 2.48% | 3.45% |

- Name + record is better in 5 of 5 splits.
- AUC rises from 0.992 to 0.998.
- Record features alone: 25.7% at 99% recall, so the names remain the core.

**Where the gain comes from (99% recall).**

| Group | name only | name + record |
| --- | --- | --- |
| birth date on both sides | 1.64% | 0.50% |
| birth date missing | 4.95% | 3.44% |
| identical-name negatives | 67.0% | 49.6% |
| no record info at all | 34.6% | 29.0% |

**Missingness carries part of the gain.** In pairs with **no record information at all**, the false alarms still fall from 34.6% to 29.0%. In that group every record feature has the same value, so the drop comes from the model treating "fields missing" as a signal, and partly from the threshold moving (section 5b).
- The presence pattern alone reaches AUC 0.86.
- Value-based gain: **0.56–1.13 pp** at 99% recall (random vs "different" stand-in, section 5e); the rest up to 1.38 pp is missingness.

**Country carries much of its signal through presence** (value strength 0.66, presence 0.73; pairs with country missing are 94% positive). Without the missing information it still helps a little (+0.09 pp when left out), so it stays. Address works through its values.

**Birth-date typos:** day/month typos and swaps are real (95–98% positive) but rare (about 570 pairs); year "typos" are different people. Version 4 adds only the real ones: false alarms unchanged, but far fewer of these true matches missed (4 instead of 12 over 5 splits, section 5f).

**Topics stay out:** they are assigned per list (source-list leak) and are weak anyway.

**Decisions, implemented in `src/screening/record_features.py` (2026-11-25):**
- Keep `r_dob_status` (as version 4: with `swapped` and `day_month_typo`), `r_country_status`, `r_id_status`, `r_address_sim`.
- Drop `r_dob_year_gap` (section 5d) and topics.
- Main result with the "different" stand-in for missing values (2.73% at 99% recall), reported as the range 2.48% (with missing info) to 3.30% (random stand-in).